# 07 · XGBoost RET: momentum, volume and opening gap

Train **15 inputs: eight momentum + six volume + one opening gap**, with 100
randomized models per quarterly fit. The rolling schedule remains 12 months
training / three months testing, initially training in 2016 and forecasting
2017Q1 through the last partial 2026Q4 ending 2026-10-30.

Fourteen inputs are available at the previous completed close. The last input
uses the current trade-day opening-price proxy. Observing that opening price,
ranking stocks and filling at the same price is an **optimistic research
assumption**. No one-minute or delayed-execution analysis is performed.

All models, data and executed outputs remain private under Git-ignored `data/`.
The original stock/date cohort and labels are preserved. Opening-gap NaNs use
XGBoost's native missing-value handling; no forecast is discarded or filled with
a made-up zero gap. The eight-input uncapped run is the comparison baseline.

**Workspace review:** source notebooks contain no private outputs. Model training and feature rebuilding are disabled by default. Read `docs/AUDIT_FINDINGS.md` before interpreting results.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json, os, sys, subprocess
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "scripts/train_xgb_ret_ensemble.py").is_file())
sys.path.insert(0,str(PROJECT_DIR / "scripts"))
FEATURE_DIR=PROJECT_DIR / "data/features/06_momentum_volume_overnight_v1"
RUN_DIR=PROJECT_DIR / "data/models/07_xgb_ret_volume_overnight100_v1"
BASELINE_DIR=PROJECT_DIR / "data/models/05_xgb_ret_network_features100_uncapped_v1"
REPORT_DIR=PROJECT_DIR / "data/reports/07_train_xgb_ret_volume_overnight"
REPORT_DIR.mkdir(parents=True,exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR",str(REPORT_DIR / ".matplotlib"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
from IPython.display import display, Markdown
from build_volume_overnight_features import MODEL_FEATURES
from ret_backtest import performance
RUN_TRAINING=False  # Inspection is the default; a completed run is never refitted.
HARD_STOP_UTC=None  # No default wall-clock cutoff.
pd.set_option("display.max_columns",24)
print("Private run:",RUN_DIR)

## Inputs and information timing

The eight momentum values are copied unchanged. Six previous-close additions:
relative daily dollar activity, relative closing-hour activity, three-versus-seven
session activity, one- and five-session signed dollar activity, and close versus
session VWAP. Raw price and volume are paired for notional proxies. Calendar
windows respect early closes and retain missing scheduled dates.

The opening gap is adjusted current open / prior adjusted close - 1, masked when
opening provenance is incomparable. Only the opening field enters it; no opening
bar high/low/close/volume or later same-day source tag is used. Published opening
bars ordinarily arrive five minutes later. This benchmark assumes that opening
field is known instantly and permits same-price entry; it is not execution evidence.

In [ ]:
manifest=json.loads((FEATURE_DIR / "MANIFEST.json").read_text())
assert manifest["model_features"]==MODEL_FEATURES and len(MODEL_FEATURES)==15
assert manifest["summary"]["forecast_cohort_identical"]
display(pd.DataFrame({"input":MODEL_FEATURES}))
display(pd.DataFrame([{"feature":k,"definition":v} for k,v in manifest["volume_definitions"].items()]))
display(pd.DataFrame.from_dict(manifest["information_timing"],orient="index",columns=["timing"]))
display(pd.read_parquet(FEATURE_DIR / "feature_availability.parquet"))

## Complete ensembles and stopping controls

Each quarterly member has a fresh random historical 80/20 row split. Train on
80%, use 20% for validation, predict with the best iteration and average all
100 probability vectors before RET. There is no subsequent full-pool refit.

Training stops after **30 validation rounds without improvement**. There is no
boosting-round maximum, fixed training RAM ceiling or default wall-clock cutoff.
Ten workers match the ten CPU cores. New batches wait when system-available
memory is below 2 GiB. Every completed model is independently checkpointed.
The initial actual-concurrency benchmark estimates runtime and RAM.

Class-mean payoffs, tree settings and equal-weight N=2/3 portfolios remain the
same as the previous experiment. Unknown selected outcomes stay unresolved.

In [ ]:
if RUN_TRAINING:
    command=[sys.executable,str(PROJECT_DIR / "scripts/train_xgb_ret_ensemble.py"),
        "--project",str(PROJECT_DIR),"--run-dir",str(RUN_DIR),"--workers","10"]
    if HARD_STOP_UTC is not None:
        command += ["--hard-stop-utc",HARD_STOP_UTC]
    subprocess.run(command,cwd=PROJECT_DIR,check=True)
status=json.loads((RUN_DIR / "RUN_STATUS.json").read_text()) if (RUN_DIR / "RUN_STATUS.json").exists() else {}
progress=json.loads((RUN_DIR / "PROGRESS.json").read_text()) if (RUN_DIR / "PROGRESS.json").exists() else {}
benchmark=json.loads((RUN_DIR / "BENCHMARK.json").read_text()) if (RUN_DIR / "BENCHMARK.json").exists() else {}
display(pd.DataFrame.from_dict(status or progress,orient="index",columns=["run_status"]))
display(pd.DataFrame.from_dict(benchmark,orient="index",columns=["benchmark"]))
COMPLETE=(RUN_DIR / "RESULTS.json").exists()
if not COMPLETE:
    display(Markdown("**The 100-member experiment is incomplete. Performance is withheld below.**"))

In [ ]:
if COMPLETE:
    new_config=json.loads((RUN_DIR / "CONFIG.json").read_text())
    old_config=json.loads((BASELINE_DIR / "CONFIG.json").read_text())
    assert new_config["parameters"]==old_config["parameters"]
    assert new_config["max_rounds"] is None
    assert new_config["early_stopping_patience"]==30
    assert new_config["members_per_quarter"]==old_config["members_per_quarter"]==100
    assert new_config["model_features"]==MODEL_FEATURES
    panel=json.loads((RUN_DIR / "PANEL.json").read_text())
    assert panel["forecast_cohort_identical"]
    new_members=pd.concat([pd.read_parquet(p) for p in sorted((RUN_DIR / "folds").glob("*/members_summary.parquet"))])
    old_members=pd.concat([pd.read_parquet(p) for p in sorted((BASELINE_DIR / "folds").glob("*/members_summary.parquet"))])
    paired=new_members.merge(old_members,on=["fold_id","member"],suffixes=("_new","_old"))
    assert len(paired)==4000
    for field in ["split_seed","booster_seed","split_train_index_sha256","split_validation_index_sha256","prediction_row_key_sha256"]:
        assert paired[field+"_new"].equals(paired[field+"_old"]),field
    print("Verified: all 4,000 member split/forecast fingerprints match the original experiment.")
    display(new_members[["best_rounds","rounds_examined","best_inner_logloss","elapsed_seconds"]].describe())

## Conditional common-date comparison

Compare both feature sets and both N values on the **same observed dates**.
Missing selected outcomes remain unknown, without replacements or invented zero
returns. Prior-data cash dates remain zero. Curves below are conditional
common-date reconstructions, not certified complete live equity curves.

The combined model has opening-time information and an ideal same-open fill;
the baseline forecasts from the previous close. Results therefore measure an
optimistic incremental-information experiment. They do not demonstrate that
the opening gap can be captured at the printed opening price.

In [ ]:
if COMPLETE:
    books=[]
    for label,path in [("momentum8",BASELINE_DIR),("combined15",RUN_DIR)]:
        frame=pd.read_parquet(path / "portfolio_daily.parquet")
        frame["feature_set"]=label
        books.append(frame)
    books=pd.concat(books,ignore_index=True)
    wide=books.pivot(index="trade_date",columns=["feature_set","n_per_side"],values="gross_return")
    common=wide.notna().all(axis=1)
    common_dates=common[common].index
    records=[]
    for (label,n),book in books.groupby(["feature_set","n_per_side"]):
        book=book.set_index("trade_date").sort_index()
        observed=book.loc[common_dates]
        for cost in [0,1,5,10]:
            records.append(dict(feature_set=label,n_per_side=n,cost_bps_per_side=cost,
                calendar_sessions=len(book),common_omitted_dates=int((~common).sum()),
                unresolved_days=int(book.gross_return.isna().sum()),
                cash_days=int(book.selected_positions.eq(0).sum()),
                metric_scope="four_way_common_observed_dates_conditional",
                **performance(observed.gross_return-observed.trade_notional*cost/10000)))
    comparison=pd.DataFrame(records)
    comparison.to_parquet(REPORT_DIR / "four_way_common_comparison.parquet",index=False)
    common.rename("included").to_frame().to_parquet(REPORT_DIR / "common_date_mask.parquet")
    display(comparison[["feature_set","n_per_side","cost_bps_per_side","sessions","common_omitted_dates","annual_mean","annual_vol","sharpe","annual_compound","max_drawdown"]])
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,cost in zip(axes,[0,1]):
        for (label,n),book in books.groupby(["feature_set","n_per_side"]):
            b=book.set_index("trade_date").loc[common_dates]
            equity=(1+b.gross_return-b.trade_notional*cost/10000).cumprod()
            ax.plot(equity.index,equity,label=f"{label}, N={n}",linewidth=1)
        ax.set_title(f"Conditional common-date equity, {cost} bp/side")
        ax.legend(fontsize=7)
        ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "conditional_feature_comparison.png",dpi=140)
    plt.show()

In [ ]:
if COMPLETE:
    results=json.loads((RUN_DIR / "RESULTS.json").read_text())
    old_results=json.loads((BASELINE_DIR / "RESULTS.json").read_text())
    display(pd.DataFrame({"momentum8":old_results["classification"],
        "combined15":results["classification"]}))
    positions=pd.read_parquet(RUN_DIR / "positions.parquet")
    unresolved=positions[positions.future_oc_return.isna()]
    unresolved.to_parquet(REPORT_DIR / "unresolved_selected_positions.parquet",index=False)
    display(unresolved[["trade_date","ticker","security_id","n_per_side","side","outcome_status"]])
    summary=dict(status="complete",total_members=4000,cohort_identical=True,
        common_observed_dates=len(common_dates),omitted_dates=int((~common).sum()),
        classification=results["classification"],run_status=status,
        comparison=comparison.to_dict("records"))
    (REPORT_DIR / "SUMMARY.json").write_text(json.dumps(summary,indent=2,default=str)+"\n")

Feature construction, membership, adjustment vintages, borrow availability,
opening/closing price proxies and missing outcomes retain their recorded
limitations. Any apparent gap reversal can include shared opening-price noise
in the feature and target. One-minute/delayed-execution work is deferred.

See `../docs/RESEARCH_CONTRACT.md` for feature formulas and clocks.

## Inherited preprocessing

The original eight momentum values are unchanged, including the previously
documented nonbinding compressed five-sigma clipping. Added volume and opening
gap variables are not newly clipped, and realized portfolio returns are never
clipped. No network propagation or OLS model is introduced.

## Feature importance across all 4,000 models

Read the saved best-iteration trees: **no training is restarted**. Sum split gain
for each feature within a model, normalize that model to 100%, then equally
average all 100 members in each of the 40 quarterly fits. A short model and a
long model have the same weight. Lines show the 10th–90th percentile across
quarters, **not confidence intervals**. The heatmap shows temporal stability.

This gain concerns the classifier's **training loss**. It is not out-of-sample
explained return variance, a directional coefficient, an ablation result or a
causal effect. Correlated variables may substitute for one another.

In [ ]:
if COMPLETE:
    from analyze_volume_overnight_results import analyze
    from IPython.display import Image
    def formatted_table(frame, formats):
        result=frame.copy()
        for column, spec in formats.items():
            result[column]=result[column].map(spec.format)
        return result
    DIAGNOSTIC_DIR=analyze(PROJECT_DIR)  # Cached extraction; reads saved models only.
    diagnostic_summary=json.loads((DIAGNOSTIC_DIR / "SUMMARY.json").read_text())
    importance=pd.read_parquet(DIAGNOSTIC_DIR / "feature_importance.parquet")
    display(formatted_table(importance[["label","group","mean_gain_share","quarter_p10","quarter_p90","mean_split_share"]],{
        "mean_gain_share":"{:.2%}","quarter_p10":"{:.2%}","quarter_p90":"{:.2%}","mean_split_share":"{:.2%}"}))
    group_importance=pd.Series(diagnostic_summary["group_gain_shares"],name="training_gain_share").to_frame()
    display(formatted_table(group_importance,{"training_gain_share":"{:.2%}"}))
    display(Image(filename=str(DIAGNOSTIC_DIR / "feature_importance.png")))

In [ ]:
if COMPLETE:
    display(Image(filename=str(DIAGNOSTIC_DIR / "feature_importance_by_quarter.png")))

## Out-of-sample explained variance and $R^2$

Use the saved **100-member averaged RET forecast**, without fitting a calibration
on evaluation outcomes. Compare the eight- and fifteen-input models on exactly
the same observed stock-days. This is the complete scored stock universe, not
only top/bottom positions and not the four-portfolio common-date subset above.
Missing outcomes remain unknown and are excluded from scoring, never zero-filled.

For realized intraday return $y$ and frozen-quarter RET forecast $\hat y$:

$$\mathrm{EV}=1-\frac{\mathrm{Var}(y-\hat y)}{\mathrm{Var}(y)},\qquad
R^2=1-\frac{\mathrm{Mean}[(y-\hat y)^2]}{\mathrm{Var}(y)}.$$

EV ignores a constant forecast bias; $R^2$ penalizes it. Negative values remain
visible. These return metrics differ from the ten-class log loss used in fitting.
The within-day version first subtracts the day's observed-universe mean from
**both** returns and forecasts, then pools those cross-sectional residuals.
That diagnostic removes common daily market moves; it does not use today's
realized mean to modify any portfolio or actual forecast.

In [ ]:
if COMPLETE:
    display(pd.DataFrame([{k:diagnostic_summary[k] for k in [
        "forecast_rows","observed_stock_days","unresolved_stock_days","observed_dates","first_date","last_date"]}]))
    return_fit=pd.read_parquet(DIAGNOSTIC_DIR / "explained_variance.parquet")
    display(formatted_table(return_fit,{"explained_variance":"{:+.4%}","r2":"{:+.4%}",
        "target_std_bps":"{:.2f}","forecast_std_bps":"{:.2f}","residual_std_bps":"{:.2f}",
        "mean_error_bps":"{:+.2f}","rmse_bps":"{:.2f}"}))
    display(Image(filename=str(DIAGNOSTIC_DIR / "out_of_sample_explained_variance.png")))

In [ ]:
if COMPLETE:
    by_year=pd.read_parquet(DIAGNOSTIC_DIR / "explained_variance_by_year.parquet")
    display(formatted_table(by_year[["feature_set","scope","year","stock_days","explained_variance","r2"]],{
        "explained_variance":"{:+.4%}","r2":"{:+.4%}"}))
    display(Image(filename=str(DIAGNOSTIC_DIR / "explained_variance_by_year.png")))

## Forecast ranking and return calibration

Rank each day's stocks by their saved RET score, form ten groups, average stocks
within each daily group, then average those daily means with equal day weights.
Decile 1 is the lowest RET forecast and decile 10 the highest. The line shows
predicted returns and bars show realized open-to-close returns on the same rows.
This broad-universe diagnostic does not replace the N=2/3 portfolio comparison.

All charts retain the opening-information and same-price execution assumptions.
The opening gap and intraday target can share opening-price noise. Gain shares
cannot attribute the incremental performance separately to volume and the gap;
that would require further controlled experiments. These periods have already
been used for development and are not a new final holdout. One-minute execution
analysis remains deferred.

In [ ]:
if COMPLETE:
    display(Image(filename=str(DIAGNOSTIC_DIR / "out_of_sample_return_calibration.png")))
    print("Private figures and numerical tables:",DIAGNOSTIC_DIR)